In [1]:
import numpy as np
import pandas as pd
from bokeh.io import output_notebook, show
from bokeh.models import ColumnDataSource
from bokeh.plotting import figure
import os

os.environ["BOKEH_MINIFIED"] = "false"
output_notebook()

Loading BokehJS ...

# Bokeh と JavaScript

Bokeh の描画と対話の機能は、ブラウザで動く JavaScript のライブラリが受け持っています。Python で図を作るときに、JavaScript のコードを足せば、対話の機能を増やせます。この章ではその書き方を説明します。

ブラウザでデバッグしやすくするために、Bokeh を読み込む前に環境変数 `BOKEH_MINIFIED` を `"false"` にしておきます。`output_notebook()` が圧縮していない版を読むので、コードを追いやすくなります。

```python
import os
from bokeh.io import output_notebook

os.environ["BOKEH_MINIFIED"] = "false"
output_notebook()
```

次の JavaScript を Notebook で実行し、出力に `ColumnDataSource` が出れば、圧縮していない版が読まれています。`Bokeh.Models(名前)` で、その名前のモデルのクラスが取れます。

この章では、次の順に説明します。

* ブラウザでのデバッグのしかた
* イベントのコールバック(`js_on_event()`)
* データの変換器(`dodge()`、`jitter()`、カラーマップ)
* データソースの絞り込み(フィルター)
* 目盛りの文字、対話ツール

In [2]:
%%javascript
let ColumnDataSource = Bokeh.Models["ColumnDataSource"];
element.append(ColumnDataSource.name)

<IPython.core.display.Javascript object>

## JavaScript のデバッグ

まず、ブラウザで JavaScript をデバッグする方法です。JavaScript から図のオブジェクトを簡単に取れるように、図を作るときに `id` で図の ID を、図形を作るときに `name` で名前を付けておきます。

In [3]:
from bokeh.models import ColumnDataSource

source = ColumnDataSource(data=dict(x=[1, 2, 3], y=[1, 3, 2]))
fig = figure(height=250)
fig._id = "my_fig"
line = fig.line("x", "y", source=source, name="my_line")
show(fig)

Notebook で上のプログラムを実行したら、ブラウザで [Ctrl+Shift+I] を押して開発者ツールを開きます。Console のタブで、次の JavaScript を実行します。折れ線の x と y のデータが出ます。変数はグローバルなので、Console から `fig`、`line`、`source` などの中身を見られます。

```javascript
fig = Bokeh.index["my_fig"].model; //❶
line = fig.select_one("my_line");  //❷
source = line.data_source;         //❸
console.log(source.data);
```

❶ JavaScript では、図は `Bokeh.index` の中の `PlotView` に対応し、その `model` が Python の `fig` にあたります。❷ `select_one()` で、`name` が `"my_line"` のモデルを取ります。Python の `line`(`GlyphRenderer`)に対応します。❸ その `data_source` が、Python の `source`(`ColumnDataSource`)にあたります。

このように、Bokeh は JavaScript の中にも Python と同じ構造のオブジェクトを作ります。

![](images/bokeh-javascript-debug.png)

## `js_on_event()` のコールバック

`Model` には `js_on_change()` と `js_on_event()` があります。`js_on_change()` は属性が変わったとき、`js_on_event()` は対話のイベントが起きたときに、JavaScript を呼びます。コールバックの中では `cb_obj` からイベントの情報を取れます。イベントのクラスは `bokeh.events` にあり、`cb_obj` の属性はそのイベントの属性と同じです。

たとえば、ダブルクリックの `DoubleTap` を見ると、`cb_obj` には `x`、`y`、`sx`、`sy` の 4 つがあります。`x`、`y` がデータの座標、`sx`、`sy` が画面のピクセルです。

In [4]:
from bokeh import events

print(events.DoubleTap.__doc__)

Announce a double-tap or double-click event on a Bokeh plot.

Attributes:
    sx (float) : x-coordinate of the event in *screen* space
    sy (float) : y-coordinate of the event in *screen* space
    x (float) : x-coordinate of the event in *data* space
    y (float) : y-coordinate of the event in *data* space




次の例では、マウスで曲線を描けます。図の上でダブルクリックすると描き始め、もう一度ダブルクリックするまで、マウスの軌跡を記録します。`DoubleTap` と `MouseMove` のコールバックで、データソースに点を足します。

❶ `source` は軌跡を入れるデータソースで、その `tags` に色のリストと、いま描いているかどうかを入れます。このデータソースには、いくつもの図形がつながっています。❷ `point_source` はマウスの今の位置を入れるデータソースで、散布図がつながっています。❸ 2 つのイベントのコールバックは、どちらも `callback` です。❹ ダブルクリックのときは `event_name` が `'doubletap'` なので、描くかどうかを切り替えます。❺ 描いている間は、マウスの位置を軌跡に足します。

In [5]:
from bokeh.models import CustomJS
from bokeh.events import MouseMove, DoubleTap
from bokeh.palettes import Category20_20

fig = figure(
    active_drag=None,
    x_range=(-10, 10),
    y_range=(-10, 10),
    frame_width=500,
    frame_height=300,
)
source = ColumnDataSource(
    data=dict(xs=[], ys=[], colors=[]), tags=[dict(colors=Category20_20, mode="normal")]
)  # ❶
pointer_source = ColumnDataSource(data=dict(x=[0], y=[0]))  # ❷

lines = fig.multi_line(
    "xs", "ys", line_color="colors", source=source, line_width=2
)  # ❶
pointer = fig.scatter(
    "x", "y", source=pointer_source, visible=False, marker="cross", size=10, color="red"
)  # ❷

callback = CustomJS(  # ❸
    args=dict(source=source, pointer=pointer),
    code="""
let data = source.data;
let info = source.tags[0];

if(cb_obj.event_name == 'doubletap'){
    if(info.mode == 'normal'){ //❹
        let colors = info.colors;
        data.xs.push([]);
        data.ys.push([]);
        data.colors.push(colors[data.xs.length % colors.length]);
        pointer.visible = true;
        pointer.data_source.change.emit();
        info.mode = 'draw';
    }
    else{
        info.mode = 'normal';
        pointer.visible = false;
    }
}
else if(cb_obj.event_name == 'mousemove'){ //❺
    pointer.data_source.data["x"] = [cb_obj.x];
    pointer.data_source.data["y"] = [cb_obj.y];
    if(info.mode == 'draw'){
        let xs = data.xs;
        let ys = data.ys;
        xs[xs.length-1].push(cb_obj.x);
        ys[ys.length-1].push(cb_obj.y);
        source.change.emit();
        pointer.data_source.change.emit();
    }
}
""",
)

fig.js_on_event(DoubleTap, callback)
fig.js_on_event(MouseMove, callback)
show(fig)

## データの変換器

データがそのままでは表示に向かないことがよくあります。そのときは、Python で変換するか、JavaScript の変換器(`Transform`)を使います。数値を色にしたり、点が重ならないようにずらしたりするのは、JavaScript でできます。

次の例の `dodge("y", 1)` は、`source` の `"y"` 列に 1 を足したものを、赤い曲線の y として使います。

In [6]:
from bokeh.transform import dodge

x = np.linspace(0, 2 * np.pi, 100)
y = np.sin(x)
source = ColumnDataSource(data=dict(x=x, y=y))
fig = figure(frame_width=400, frame_height=250)
fig.line("x", "y", source=source)
line = fig.line("x", dodge("y", 1), color="red", source=source)
show(fig)

変換器を使うとき、図形の `y` は列名の文字列ではなく、変換を表す辞書になります。これは、`source` の `y` 列に `Dodge` をかけるという意味です。`Dodge` が変換器のモデルで、`dodge()` はそれを作る関数です。

In [7]:
line.glyph.y

Field(field='y', transform=Dodge(id='p1153', ...), units=Unspecified)

点の座標がまったく同じだと、散布図では分布が分かりません。そこで `jitter()` で座標に少しゆらぎを足します。次のプログラムは `np.round()` で乱数を丸めているので、点は格子の上に並びます(左)。`jitter()` で x と y にゆらぎを足すと、分布が分かります(右)。

In [8]:
from bokeh.transform import jitter
from bokeh.layouts import row

x, y = np.round(np.random.normal(scale=0.2, size=(2, 500)), 1)

source = ColumnDataSource(data=dict(x=x, y=y))
fig1 = figure(frame_width=250, frame_height=250)
fig1.scatter("x", "y", source=source, alpha=0.3)
fig2 = figure(frame_width=250, frame_height=250)
fig2.scatter(
    jitter("x", 0.1, distribution="normal"),
    jitter("y", 0.1, distribution="normal"),
    source=source,
    alpha=0.3,
)
layout = row(fig1, fig2)
show(layout)

次の例では、データソースの値で点の色を決めます。

❶ `linear_cmap()` は、数値を色にする変換器を作ります。列名のほかの 3 つの引数は、色のリスト、最小値、最大値です。`z` 列の値を `Viridis` の色にします。❷ 点の `fill_color` にその変換器を入れます。❸ `ColorBar` を使うと、値と色の対応を出せます。その `color_mapper` が、変換の `ColorMapper` です。

In [9]:
from bokeh.transform import linear_cmap
from bokeh.palettes import Viridis
from bokeh.models import ColorBar

x, y = np.random.normal(scale=0.2, size=(2, 500))
z = (x**2 + y**2) ** 0.5

source = ColumnDataSource(data=dict(x=x, y=y, z=z))
fig_cmap1 = figure(frame_width=400, frame_height=300)
cmap = linear_cmap("z", Viridis[256], z.min(), z.max())  # ❶
c = fig_cmap1.scatter(
    "x", "y", fill_color=cmap, line_color=None, source=source, alpha=1, size=6
)  # ❷
colorbar = ColorBar(
    color_mapper=cmap["transform"],  # ❸
    label_standoff=12,
    border_line_color=None,
    location=(0, 0),
)
fig_cmap1.add_layout(colorbar, "right")

`linear_cmap()` は連続の値を連続の色にします。一方、`factor_cmap()` は分類の値を色にします。列名のほかの 2 つの引数は、カラーマップと分類のリストで、分類の順番で色を選びます。

❶ `iris.csv` を読むときに、`species` 列を分類型にします。❷ 分類型の列は `cat.categories` で、すべての分類が取れます。こうすると、1 つの散布図で分類ごとの分布を出せます。

In [10]:
from bokeh.transform import factor_cmap
from bokeh.palettes import Category10

df = pd.read_csv("data/iris.csv", dtype={"species": "category"})  # ❶

source = ColumnDataSource(data=df)
fig_cmap2 = figure(frame_width=400, frame_height=300)
cmap = factor_cmap("species", Category10[10], df.species.cat.categories)  # ❷
c = fig_cmap2.scatter(
    "sepal_length",
    "sepal_width",
    fill_color=cmap,
    legend_field="species",
    line_color=None,
    source=source,
    alpha=1,
    size=6,
)

ここまでの変換器の JavaScript は Bokeh に入っているので、自分で書く必要はありません。自分で書きたいときは `CustomJSTransform` を使います。`func` は 1 つの値を、`v_func` は配列を変換します。`func` では `x` が変換する値で、戻り値が変換の結果です。`v_func` では `xs` が配列で、戻り値が変換した配列です。

次の例では、❶ `CustomJSTransform` を作り、$f(x) = \sin(x) \cdot \sin(10 x)$ を計算します。列のデータをまとめて変換するので、`func` は書きません。❷ `transform()` で、`x` 列にこの変換器をかけます。

In [11]:
layout_cmap = row(fig_cmap1, fig_cmap2)
show(layout_cmap)

In [12]:
from bokeh.transform import transform
from bokeh.models import CustomJSTransform

x = np.linspace(0, 2 * np.pi, 1000)

source = ColumnDataSource(data=dict(x=x))
fig_cjs1 = figure(frame_width=400, frame_height=250)

transform_sin = CustomJSTransform(  # ❶
    func="return Math.sin(x) * Math.sin(10*x);",
    v_func="return xs.map(x=>Math.sin(x) * Math.sin(10*x));",
)

fig_cjs1.line("x", transform("x", transform_sin), source=source);  # ❷

`v_func` の `xs` は、その列のデータだけです。`args` にほかのモデルを渡すと、`v_func` からも使えます。次の例では `source` を `args` に入れ、`x` 列と `y` 列の二乗和の平方根を返して、点の大きさにします。

In [13]:
x, y = np.random.normal(scale=0.2, size=(2, 500))
source = ColumnDataSource(data=dict(x=x, y=y))
fig_cjs2 = figure(frame_width=400, frame_height=300)

transform_dist = CustomJSTransform(
    args=dict(source=source),
    v_func="""
        let data = source.data;
        let ys = data.y;
        return xs.map((x, i) => {
            let y = ys[i];
            return 10 * Math.sqrt(x*x + y*y);
        })
    """,
)

c = fig_cjs2.scatter(
    "x",
    "y",
    size=transform("x", transform_dist),
    line_color=None,
    source=source,
    alpha=1,
)

In [14]:
layout_cjs = row(fig_cjs1, fig_cjs2)
show(layout_cjs)

## データソースの絞り込み

JavaScript で、データソースのデータを絞り込めます。次の例では、ドロップダウンで分類を選ぶと、その分類の点だけが出ます。

❶ ドロップダウンの `options` に `species` 列の値を入れ、最初の値を選んでおきます。❷ `GroupFilter` は、ある列が値と等しいデータだけを残します。`column_name` が列名、`group` が値です。❸ フィルターは `CDSView` の `filters` に入れ、`source` に絞り込むデータソースを入れます。❹ 図形を作るときに `view` でこのビューを渡します。❺ ドロップダウンの値が変わると `callback` が走ります。❻ `cb_obj.value` が選んだ値なので、それをフィルターの `group` に入れます。❼ 最後に `change.emit()` で図を描き直します。

In [15]:
from bokeh.layouts import column
from bokeh.models import CDSView, GroupFilter, Select

df = pd.read_csv("data/iris.csv", dtype={"species": "category"})

categories = df.species.cat.categories.tolist()
select = Select(options=categories, value=categories[0])  # ❶
source = ColumnDataSource(data=df)
group_filter = GroupFilter(column_name="species", group=select.value)  # ❷
view = CDSView(filter=group_filter)  # ❸

fig_filter1 = figure(frame_width=400, frame_height=300)
c = fig_filter1.scatter(
    "sepal_length",
    "sepal_width",
    line_color=None,
    source=source,
    view=view,
    alpha=1,
    size=6,
)  # ❹

callback = CustomJS(
    args=dict(filter=group_filter, source=source),
    code="""
    filter.group = cb_obj.value; //❻
    source.change.emit();        //❼
    """,
)

select.js_on_change("value", callback)  # ❺
layout_filter1 = column(select, fig_filter1)

`CustomJSFilter` なら、自分のコードで絞り込めます。`args` と `code` の使い方は `CustomJS` と同じです。戻り値は、整数か真偽値の配列で、NumPy の索引と同じように使われます。

次の例では、❶ `RangeSlider` で `petal_length` の範囲を選びます。❷ スライダーの値を読めるように、`args` にスライダーを入れます。❸ コードの中では、`source` が絞り込むデータソースです。❹ 列の各値について、範囲に入るかを調べます。`Array.from(data)` で JavaScript の配列にしてから `map()` で真偽値の配列を作ります。

In [16]:
from bokeh.models import CustomJSFilter, RangeSlider

source = ColumnDataSource(data=df)
fig_filter2 = figure(frame_width=400, frame_height=300)
slider = RangeSlider(  # ❶
    start=df.petal_length.min(),
    end=df.petal_length.max(),
    value=(df.petal_length.min(), df.petal_length.max()),
    step=0.1,
    title="petal length",
)

func_filter = CustomJSFilter(  # ❷
    args=dict(slider=slider),
    code="""
    let data = source.data['petal_length']; //❸
    let [start, end] = slider.value;
    let res = Array.from(data).map(v => start <= v && v <= end); //❹
    return res;
    """,
)

view = CDSView(filter=func_filter)

c = fig_filter2.scatter(
    "sepal_length",
    "sepal_width",
    fill_color=linear_cmap(
        "petal_length", Viridis[256], df.petal_length.min(), df.petal_length.max()
    ),
    line_color=None,
    source=source,
    view=view,
    alpha=1,
    size=6,
)

callback = CustomJS(args=dict(source=source), code="source.change.emit();")

slider.js_on_change("value", callback)
layout_filter2 = column(slider, fig_filter2)

In [17]:
layout_filter = row(layout_filter1, layout_filter2)
show(layout_filter)

## 目盛りの文字を変える

`FuncTickFormatter` は、JavaScript で目盛りの文字を作ります。次の例では、いくつかの言い回しの確率分布を比べます。y 軸の目盛りが言い回しで、曲線はその分布の推定です。

❶ Bokeh のサンプルデータから `probly` を読みます。各列が言い回しの確率の評価です。❷ `gaussian_kde()` で、各列の分布を推定します。比べやすくするために、曲線の最大値を 2 にそろえます。❸ 推定の結果を 1 つの `DataFrame` にまとめ、`zscore()` で列ごとに標準化します。

In [18]:
from bokeh.sampledata.perceptions import probly  # ❶
from scipy.stats import gaussian_kde
import numpy as np

x = np.linspace(0, 100, 200)
df_kde = probly.apply(lambda s: gaussian_kde(s)(x)).set_index(x)  # ❷
df_kde = df_kde * (2 / df_kde.max())  # ❸

❹ ループで列ごとに曲線を描き、y の値は `dodge()` でずらします。❺ `FixedTicker` で y の目盛りの位置を決めます。❻ `FuncTickFormatter` で目盛りの文字を作ります。`tick` が目盛りの位置なので、それを使って `labels` から文字を選びます。

In [19]:
from bokeh.models import FixedTicker, CustomJSTickFormatter
from bokeh.palettes import Category20_20

source = ColumnDataSource(df_kde)
fig_tick = figure()

for i, col in enumerate(df_kde.columns):
    fig_tick.line(
        "index",
        dodge(col, i),
        source=source,
        color=Category20_20[i],
        line_width=2,
        alpha=0.6,
    )  # ❹

fig_tick.yaxis.ticker = FixedTicker(ticks=np.arange(df_kde.shape[1]))  # ❺
fig_tick.yaxis.ticker.tags = ["x", "y", "z"]
fig_tick.yaxis.formatter = CustomJSTickFormatter(
    args=dict(labels=list(df_kde.columns)), code="return labels[tick];"
)  # ❻
show(fig_tick)

## 対話ツール

ツールバーには、JavaScript を呼べるツールが 3 つあります。`CustomAction` はボタンを押したとき、`TapTool` は図形をクリックしたとき、`HoverTool` は図形の上にマウスが来たときに動きます。

次の例では、`HoverTool` で中国の各省の説明を出します。省の上にマウスを置くと、右の欄に説明が出ます。

❶ `china_simple.json` は省の形の GeoJSON です。`GeoJSONDataSource` がそのまま読めます。❷ `china_info.json` は省ごとの説明で、`json.load()` で辞書にします。❸ `patches()` で省の形を描き、色は `population` 列で決めます。❹ `Div` に説明を出し、`tags` に省の説明の辞書を入れます。Python の辞書は JavaScript のオブジェクトに変換されます。❺ `HoverTool` の `renderers` に、反応する図形を入れます。

❻ コールバックの `cb_data.index.indices` が、マウスの下の図形の番号です。空でなければ最初の番号を使います。❼ `name` 列から省の名前を取り、❽ 説明の辞書から文を引いて、`div_info.text` に入れます。

In [20]:
import json
from bokeh.models import GeoJSONDataSource, Div, CustomJS, HoverTool
from bokeh.layouts import row
from bokeh.transform import linear_cmap
from bokeh.palettes import Viridis

with open("data/china_simple.json", encoding="utf-8") as f:  # ❶
    geojson_string = f.read()

with open("data/china_info.json", encoding="utf-8") as f:  # ❷
    province_info = json.load(f)

source = GeoJSONDataSource(geojson=geojson_string)
fig = figure(match_aspect=True, aspect_scale=1, frame_width=500, frame_height=500)
patches = fig.patches(
    "xs",
    "ys",
    source=source,
    line_color="black",  # ❸
    fill_color=linear_cmap("population", Viridis[256], low=None, high=None),
)

div_info = Div(width=300, tags=[province_info])  # ❹

callback = CustomJS(
    args=dict(source=source, div_info=div_info),
    code="""
    let indices = cb_data.index.indices; //❻
    let info = div_info.tags[0];

    if(indices.length > 0){
        let name = source.data['name'][indices[0]]; //❼
        let text = info[name]; //❽
        div_info.text = `<h2>${name}</h2><p>${text}</p>`; 
    }
    """,
)

hover = HoverTool(tooltips=None, callback=callback, renderers=[patches])  # ❺
fig.tools.append(hover)
layout = row(fig, div_info)
show(layout)

## まとめ

* Bokeh の図と対話はブラウザの JavaScript が動かす。Python のモデルと同じ構造が JavaScript 側にもできる。
* `id` と `name` を付けると、開発者ツールの Console から `Bokeh.index` で触れる。
* 属性の変化は `js_on_change()`、イベントは `js_on_event()` で受ける。情報は `cb_obj` にある。
* 変換器は Bokeh に用意がある(`dodge()`、`jitter()`、`linear_cmap()`、`factor_cmap()`)。自分で書くなら `CustomJSTransform`。
* 絞り込みは `CDSView` の `filters`(`GroupFilter`、`CustomJSFilter`)で行い、図形に `view` を渡す。
* 目盛りの文字は `FuncTickFormatter`、位置は `FixedTicker` で決める。
* ツールは `CustomAction`、`TapTool`、`HoverTool` に JavaScript を付けられる。